# Track 1 Test Prediction - DeBERTa-v2-XXLarge MoCo-SCL

Use the best compact MoCo-SCL checkpoint to predict `data/test/track1.jsonl` and write the official submission file `track1.pred.jsonl`.

each line is a JSON object with `Value` as the key, aligned line-by-line with the test file.

In [1]:
from pathlib import Path
import json
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import AutoConfig, AutoModel, AutoTokenizer, DataCollatorWithPadding
from peft import PeftModel


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "yaoqiu.md").exists() and (path / "data" / "test" / "track1.jsonl").exists():
            return path
    raise FileNotFoundError("Could not locate repo root containing yaoqiu.md and data/test/track1.jsonl")


REPO_ROOT = find_repo_root()
TEST_PATH = REPO_ROOT / "data" / "test" / "track1.jsonl"
MODEL_DIR = REPO_ROOT / "outputs" / "track1" / "encoder" / "deberta_v2_xxlarge_moco_scl" / "best_macro_f1_model"
ADAPTER_DIR = MODEL_DIR / "query_encoder_lora"
HEADS_PATH = MODEL_DIR / "heads.pt"
OUTPUT_PATH = REPO_ROOT / "track1.pred.jsonl"

MAX_LENGTH = 512
BATCH_SIZE = 16

print(f"Repo root : {REPO_ROOT}")
print(f"Test file : {TEST_PATH}")
print(f"Model dir : {MODEL_DIR}")
print(f"Output    : {OUTPUT_PATH}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Repo root : /home/yangdejin/nlpcc/nlpcc_task2
Test file : /home/yangdejin/nlpcc/nlpcc_task2/data/test/track1.jsonl
Model dir : /home/yangdejin/nlpcc/nlpcc_task2/outputs/track1/encoder/deberta_v2_xxlarge_moco_scl/best_macro_f1_model
Output    : /home/yangdejin/nlpcc/nlpcc_task2/track1.pred.jsonl


In [2]:
required_paths = [TEST_PATH, MODEL_DIR, ADAPTER_DIR, HEADS_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required file(s):\n" + "\n".join(missing))

with open(MODEL_DIR / "best_metric.json", "r", encoding="utf-8") as f:
    best_metric = json.load(f)

print(json.dumps(best_metric, ensure_ascii=False, indent=2))

{
  "best_macro_f1": 0.9417667224757209,
  "global_step": 770,
  "epoch": 14.0,
  "model_name": "microsoft/deberta-v2-xxlarge",
  "num_labels": 19,
  "labels": [
    "Self-direction–thought",
    "Self-direction–action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power–dominance",
    "Power–resources",
    "Face",
    "Security–personal",
    "Security–societal",
    "Tradition",
    "Conformity–rules",
    "Conformity–interpersonal",
    "Humility",
    "Benevolence–dependability",
    "Benevolence–caring",
    "Universalism–concern",
    "Universalism–nature",
    "Universalism–tolerance"
  ]
}


In [3]:
def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class CompactMoCoSCLClassifier(nn.Module):
    def __init__(self, model_dir, adapter_dir, heads):
        super().__init__()
        model_name = heads.get("model_name", "microsoft/deberta-v2-xxlarge")
        num_labels = int(heads.get("num_labels", 19))

        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            torch_dtype = torch.bfloat16
        elif torch.cuda.is_available():
            torch_dtype = torch.float16
        else:
            torch_dtype = torch.float32

        config = AutoConfig.from_pretrained(model_dir, num_labels=num_labels)
        base_encoder = AutoModel.from_pretrained(model_name, config=config, torch_dtype=torch_dtype)
        self.encoder = PeftModel.from_pretrained(base_encoder, adapter_dir)
        self.classifier = nn.Linear(config.hidden_size, num_labels)
        self.classifier.load_state_dict(heads["classifier"])

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        pooled = mean_pooling(outputs.last_hidden_state, attention_mask)
        pooled = pooled.to(self.classifier.weight.dtype)
        return self.classifier(pooled)


heads = torch_load_compat(HEADS_PATH)
raw_id2label = heads.get("id2label") or {i: label for i, label in enumerate(best_metric["labels"])}
id2label = {int(idx): label for idx, label in raw_id2label.items()}
VALUE_LABELS = [id2label[i] for i in range(len(id2label))]

print(f"Loaded {len(VALUE_LABELS)} labels")
for idx, label in enumerate(VALUE_LABELS):
    print(f"{idx:02d}: {label}")

Loaded 19 labels
00: Self-direction–thought
01: Self-direction–action
02: Stimulation
03: Hedonism
04: Achievement
05: Power–dominance
06: Power–resources
07: Face
08: Security–personal
09: Security–societal
10: Tradition
11: Conformity–rules
12: Conformity–interpersonal
13: Humility
14: Benevolence–dependability
15: Benevolence–caring
16: Universalism–concern
17: Universalism–nature
18: Universalism–tolerance


In [4]:
def build_text(row):
    response = str(row.get("Consistent Value Response", "")).strip()
    return "Response: " + response


records = []
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if "Consistent Value Response" not in row:
            raise KeyError(f"Line {line_no} has no 'Consistent Value Response' field")
        records.append(row)

texts = [build_text(row) for row in records]
print(f"Loaded {len(records)} test examples")
print(texts[0])

Loaded 514 test examples
Response: I would embrace the opportunity to explore cutting-edge tools, as I believe novelty and risk-taking will elevate the project's impact and inspire fresh directions.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return tokenizer(self.texts[idx], truncation=True, max_length=MAX_LENGTH)


dataset = TextDataset(texts)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = CompactMoCoSCLClassifier(MODEL_DIR, ADAPTER_DIR, heads)
model.to(device)
model.eval()

pred_ids = []
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Predicting"):
        batch = {key: value.to(device) for key, value in batch.items()}
        logits = model(**batch)
        pred_ids.extend(torch.argmax(logits, dim=-1).cpu().tolist())

pred_labels = [id2label[pred_id] for pred_id in pred_ids]
print(f"Predicted {len(pred_labels)} labels")
print(Counter(pred_labels))

Using device: cuda


Loading weights:   0%|          | 0/778 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v2-xxlarge
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Predicting:   0%|          | 0/33 [00:00<?, ?it/s]

Predicted 514 labels
Counter({'Stimulation': 58, 'Conformity–rules': 53, 'Benevolence–caring': 42, 'Face': 38, 'Conformity–interpersonal': 37, 'Power–resources': 34, 'Security–personal': 28, 'Benevolence–dependability': 28, 'Achievement': 26, 'Hedonism': 26, 'Universalism–concern': 24, 'Power–dominance': 23, 'Self-direction–thought': 19, 'Self-direction–action': 17, 'Humility': 16, 'Tradition': 14, 'Universalism–nature': 11, 'Security–societal': 10, 'Universalism–tolerance': 10})


In [6]:
if len(pred_labels) != len(records):
    raise RuntimeError(f"Prediction count mismatch: {len(pred_labels)} predictions for {len(records)} records")

valid_labels = set(VALUE_LABELS)
invalid_labels = sorted(set(pred_labels) - valid_labels)
if invalid_labels:
    raise ValueError(f"Invalid predicted labels: {invalid_labels}")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for label in pred_labels:
        f.write(json.dumps({"Value": label}, ensure_ascii=False) + "\n")

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    output_lines = [json.loads(line) for line in f]

assert len(output_lines) == len(records)
assert all(list(row.keys()) == ["Value"] for row in output_lines)

print(f"Saved official Track 1 submission file: {OUTPUT_PATH}")
print(f"Line count: {len(output_lines)}")
print("First 5 lines:")
for row in output_lines[:5]:
    print(json.dumps(row, ensure_ascii=False))

Saved official Track 1 submission file: /home/yangdejin/nlpcc/nlpcc_task2/track1.pred.jsonl
Line count: 514
First 5 lines:
{"Value": "Stimulation"}
{"Value": "Power–resources"}
{"Value": "Power–resources"}
{"Value": "Achievement"}
{"Value": "Power–resources"}
